# 02 — Exercises: feeding the presales pipeline

**Block 1 — Business & Data Foundations · Lesson 02 · Exercises**

The lesson notebook built a spam filter's data. These exercises stay at **MSMK Consulting**
and move to another department: presales is overloaded during peaks, and nobody can say where
the time goes, because the evidence sits in three different systems — the same shape of
problem as your Assignment Brief.

Your job is the one that comes before any analysis: **read each source correctly, then join
them without losing a single bid.**

| # | Source | Format | What you practise | Time |
|---|---|---|---|---|
| 1 | SAP bid export | CSV | separator, decimals, dates, encoding | 5 min |
| 2 | CRM accounts | JSON | `json_normalize` on a nested object | 5 min |
| 3 | Strategic qualification | Excel | header offset, messy text | 5 min |
| 4 | All three | — | left joins that keep every row | 5 min |

### How to work

- Fill in every cell marked **`# TODO`**. The cell right after it checks your answer and
  prints `✅ Exercise N passed`, or an error message telling you what is still wrong.
- **Do not edit the check cells**, and do not edit the cells marked *given*: they build the
  raw sources and are the equivalent of the systems you cannot change in real life.
- Everything runs **offline**, in a temporary folder. Nothing is downloaded.
- Stuck? The lesson notebook has the matching section, and every exercise names it.

> **Extra library:** `openpyxl`, to read the Excel file.
> Install with `pip install openpyxl` if the setup cell complains.

## 0. Setup — *given, run it and move on*

In [1]:
import json
import tempfile
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 7
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

# Landing zone: the raw files, exactly as the systems send them. The same idea as the
# lesson, in a temporary folder that disappears when you close the notebook.
WORKDIR = Path(tempfile.mkdtemp(prefix="lesson02_ex_"))
LANDING = WORKDIR / "landing"
LANDING.mkdir()


def passed(n):
    print(f"✅ Exercise {n} passed")


print("pandas", pd.__version__)
print("landing zone:", LANDING)

pandas 3.0.6
landing zone: C:\Users\sergi\AppData\Local\Temp\lesson02_ex_m7r7tnsl\landing


## 1. The three sources — *given*

The cell below plays the part of three systems you do not control: SAP writes a
semicolon-separated export from a Spanish locale, the CRM answers with nested JSON, and
presales keep a decorated Excel sheet that a human typed by hand.

**In your Assignment Brief this cell disappears** and you read your real files instead.

In [2]:
# ---------------------------------------------------------------- the bid pipeline (ground truth)
N_BIDS = 2_000
SECTORS = ["Naval", "Energía", "Defensa", "Logística", "Infraestructura"]
STAGES = ["Elaboración", "Entregada", "Ganada", "Perdida"]
ACCOUNTS = [f"ACC{i:04d}" for i in range(120)]
OWNERS = [f"OWN{i:03d}" for i in range(25)]
start = datetime(2025, 9, 1)

bids = pd.DataFrame(
    {
        "bid_id": [f"BID{i:05d}" for i in range(N_BIDS)],
        "account_id": rng.choice(ACCOUNTS, N_BIDS),
        "owner_id": rng.choice(OWNERS, N_BIDS),
        "opened_at": [
            start + timedelta(days=int(d))
            for d in np.sort(rng.integers(0, 365, N_BIDS))
        ],
        "sector": rng.choice(SECTORS, N_BIDS),
        "cost_eur": np.round(rng.gamma(3.0, 42_000, N_BIDS), 2),
        "margin_pct": np.round(np.clip(rng.normal(18, 6, N_BIDS), -5, 45), 2),
        "stage": rng.choice(STAGES, N_BIDS, p=[0.18, 0.22, 0.29, 0.31]),
    }
)
bids.loc[rng.random(N_BIDS) < 0.06, "margin_pct"] = (
    np.nan
)  # SAP leaves the margin empty sometimes

# 1. SAP export -> CSV, Spanish locale: ";" separator, "," decimals, dd/mm/yyyy, latin-1, "N/D" for missing
bids.to_csv(
    LANDING / "sap_bids.csv",
    sep=";",
    decimal=",",
    index=False,
    date_format="%d/%m/%Y",
    encoding="latin-1",
    na_rep="N/D",
)

# 2. CRM -> JSON with a nested contact object per account
crm = [
    {
        "account_id": a,
        "name": f"Cliente {i:03d}, S.A.",
        "country": str(
            rng.choice(["ES", "PT", "FR", "IT", "MA"], p=[0.62, 0.13, 0.11, 0.09, 0.05])
        ),
        "since": int(rng.integers(2008, 2025)),
        "contact": {
            "name": f"Contacto {i:03d}",
            "email": f"contacto{i:03d}@cliente{i:03d}.com",
        },
    }
    for i, a in enumerate(ACCOUNTS)
]
(LANDING / "crm_accounts.json").write_text(
    json.dumps(crm, indent=1, ensure_ascii=False), encoding="utf-8"
)

# 3. Presales -> Excel with three decorative rows on top and hand-typed qualifications
QUAL_RAW = [
    "Estrategica",
    "estrategica ",
    " ESTRATEGICA",
    "Tactica",
    "tactica ",
    " TACTICA",
    "Oportunista",
    "oportunista ",
]
qual_bids = bids["bid_id"].sample(1_400, random_state=SEED).sort_values().tolist()
qual = pd.DataFrame(
    {
        "bid_id": qual_bids,
        "qualification": rng.choice(QUAL_RAW, len(qual_bids)),
        "reviewer": rng.choice(["A. Ruiz", "M. Soler", "J. Pena"], len(qual_bids)),
    }
)
with pd.ExcelWriter(LANDING / "qualification.xlsx", engine="openpyxl") as xw:
    qual.to_excel(xw, sheet_name="Cualificación", startrow=4, index=False)
    ws = xw.book["Cualificación"]
    ws["A1"] = "MSMK Consulting — Departamento de Preventa"
    ws["A2"] = "Cualificación estratégica de ofertas"
    ws["A3"] = "Uso interno. Actualizado semanalmente."

print("raw sources written to", LANDING)
for f in sorted(LANDING.iterdir()):
    print(f"  {f.name:22} {f.stat().st_size/1024:8.1f} KB")

raw sources written to C:\Users\sergi\AppData\Local\Temp\lesson02_ex_m7r7tnsl\landing
  crm_accounts.json          23.2 KB
  qualification.xlsx         26.9 KB
  sap_bids.csv              136.1 KB


---

## Exercise 1 — Read the SAP export without trusting it

*Lesson section 3.1 · 5 minutes*

`sap_bids.csv` was written by SAP in a Spanish locale. **Look at the raw bytes first** (the
given cell below), then read it into `bids_raw` so that nothing arrives broken:

- 2,000 rows and 8 columns — not one column of 2,000 mangled strings;
- `cost_eur` and `margin_pct` are **floats**, not text;
- `opened_at` is a **datetime**, and the earliest bid is in **September 2025** (a January
  one means the day and the month were swapped);
- `bid_id`, `account_id` and `owner_id` stay **text** — they are identifiers, never numbers;
- the missing margins arrive as `NaN`, and the accents in `Elaboración` survive.

> **Hint.** Five arguments of `pd.read_csv` do all of it: `sep`, `decimal`, `encoding`,
> `na_values`, and `dtype` for the three identifier columns. For the date, add
> `parse_dates=["opened_at"]` with `dayfirst=True`.

In [3]:
# Look before you read: the first bytes of the file, exactly as SAP wrote them.
with open(LANDING / "sap_bids.csv", "rb") as fh:
    print(fh.read(320).decode("latin-1"))

bid_id;account_id;owner_id;opened_at;sector;cost_eur;margin_pct;stage
BID00000;ACC0113;OWN015;01/09/2025;Logística;57700,4;15,83;Perdida
BID00001;ACC0075;OWN021;01/09/2025;Energía;142236,57;17,31;Ganada
BID00002;ACC0082;OWN006;01/09/2025;Energía;84481,86;N/D;Elaboración
BID00003;ACC0107;OWN021;01/09/2025;Infraestru


In [4]:
# TODO: read sap_bids.csv into `bids_raw`, correctly.
bids_raw = bids_raw = pd.read_csv(
    LANDING / "sap_bids.csv",
    sep=";",
    decimal=",",
    encoding="latin-1",
    na_values="N/D",
    parse_dates=["opened_at"],
    dayfirst=True,
    dtype={"bid_id": str, "account_id": str, "owner_id": str}
)

print(bids_raw.shape)
bids_raw.dtypes

(2000, 8)


bid_id                   str
account_id               str
owner_id                 str
opened_at     datetime64[us]
sector                   str
cost_eur             float64
margin_pct           float64
stage                    str
dtype: object

In [5]:
# --- check 1 (do not edit) ---
from pandas.api.types import is_datetime64_any_dtype, is_float_dtype, is_string_dtype

assert bids_raw.shape == (
    2_000,
    8,
), f"expected (2000, 8), got {bids_raw.shape} — check `sep`"
assert is_float_dtype(bids_raw["cost_eur"]), "cost_eur is not numeric — check `decimal`"
assert is_float_dtype(
    bids_raw["margin_pct"]
), "margin_pct is not numeric — check `decimal` and `na_values`"
assert is_datetime64_any_dtype(
    bids_raw["opened_at"]
), "opened_at is not a datetime — check `parse_dates`"
assert (
    bids_raw["opened_at"].min().month == 9
), "day and month are swapped — add `dayfirst=True`"
assert (
    bids_raw["margin_pct"].isna().sum() > 50
), "the missing margins were not recognised — check `na_values`"
for col in ["bid_id", "account_id", "owner_id"]:
    assert is_string_dtype(bids_raw[col]) and not is_float_dtype(
        bids_raw[col]
    ), f"{col} must stay text — check `dtype`"
assert "Elaboración" in set(
    bids_raw["stage"]
), "the accents are corrupted — check `encoding`"
passed(1)

✅ Exercise 1 passed


---

## Exercise 2 — Flatten the CRM payload

*Lesson section 3.2 · 5 minutes*

The CRM answers with **nested** JSON: each account carries a `contact` object, and an object
is not a table column.

Load `crm_accounts.json` and flatten it into a DataFrame called `accounts`: one row per
account, with the contact fields as their own columns, named `contact.name` and
`contact.email`.

> **Hint.** `json.loads(path.read_text(encoding="utf-8"))` gives you a list of dicts;
> `pd.json_normalize` turns it into a flat table in one call.

In [6]:
# TODO: load and flatten the CRM payload.
raw_crm = json.loads((LANDING / "crm_accounts.json").read_text(encoding="utf-8"))
accounts = pd.json_normalize(raw_crm)

print(accounts.shape)
accounts.head(3)

(120, 6)


,account_id,name,country,since,contact.name,contact.email
0,ACC0000,"Cliente 000, S.A.",ES,2018,Contacto 000,contacto000@cliente000.com
1,ACC0001,"Cliente 001, S.A.",ES,2016,Contacto 001,contacto001@cliente001.com
2,ACC0002,"Cliente 002, S.A.",ES,2012,Contacto 002,contacto002@cliente002.com


In [7]:
# --- check 2 (do not edit) ---
assert (
    isinstance(accounts, pd.DataFrame) and len(accounts) == 120
), "accounts must be a DataFrame with one row per account (120)"
assert {"contact.name", "contact.email"} <= set(
    accounts.columns
), "the contact object was not flattened — use pd.json_normalize"
assert not any(
    isinstance(v, dict) for v in accounts.iloc[0]
), "a nested object is still sitting inside a cell"
assert {"account_id", "name", "country"} <= set(
    accounts.columns
), "account_id, name and country must survive the flattening"
passed(2)

✅ Exercise 2 passed


---

## Exercise 3 — Rescue the qualification sheet

*Lesson section 3.4 · 5 minutes*

`qualification.xlsx` is a human artefact: a title, a subtitle, a disclaimer, an empty row,
and only then the table. The sheet is called **`Cualificación`**, and the `qualification`
column was typed by three different people over two years.

**3a.** Read the sheet into `quals` so that the first row of the table is the real header
(no `Unnamed: 0` columns), with `bid_id` as text.

**3b.** Add a column `qualification_clean` to `quals` holding exactly **three** values —
`estrategica`, `tactica`, `oportunista` — by removing the surrounding spaces and
lowercasing.

> **Hint.** `pd.read_excel(path, sheet_name=..., skiprows=...)`, then `.str.strip()` and
> `.str.lower()` on the column.

In [13]:
# TODO (3a): read the sheet, skipping the decoration.
quals = pd.read_excel(
    LANDING / "qualification.xlsx",
    sheet_name="Cualificación",
    skiprows=4,
    dtype={"bid_id": str}
)

# TODO (3b): add `qualification_clean` with the three tidy values.
quals["qualification_clean"] = quals["qualification"].str.strip().str.lower()

print(quals.shape)
quals["qualification_clean"].value_counts()

(1400, 4)


qualification_clean
estrategica    534
tactica        508
oportunista    358
Name: count, dtype: int64

In [14]:
# --- check 3 (do not edit) ---
assert (
    isinstance(quals, pd.DataFrame) and len(quals) == 1_400
), f"expected 1400 rows, got {len(quals)} — check `skiprows`"
assert not [
    c for c in quals.columns if str(c).startswith("Unnamed")
], "the decoration was read as data — check `skiprows`"
assert {"bid_id", "qualification", "qualification_clean"} <= set(
    quals.columns
), "quals needs bid_id, qualification and qualification_clean"
assert set(quals["qualification_clean"]) == {
    "estrategica",
    "tactica",
    "oportunista",
}, f"expected three tidy values, got {sorted(set(quals['qualification_clean']))[:6]}"
passed(3)

✅ Exercise 3 passed


---

## Exercise 4 — One table, without losing a bid

*Lesson section 5 · 5 minutes*

Three sources, one analysis table — and this is where projects quietly lose rows.

Build `analysis`, starting from `bids_raw` and adding, **with left joins**:

| From | Columns to bring | Relationship |
|---|---|---|
| `accounts` | the CRM `name`, renamed `account_name`, and `country` | many bids → one account |
| `quals` | `qualification_clean` | at most one qualification per bid |

Rules: exactly **2,000 rows** at the end, `bid_id` still unique, and no `_x` / `_y` columns.
Missing values are expected — 600 bids were never qualified, and a left join is what lets
you *see* that instead of silently dropping them.

> **Hint.** Select and rename before you merge:
> `accounts[["account_id", "name", "country"]].rename(columns={"name": "account_name"})`.
> An inner join here would throw away rows; `how="left"` never does.

In [ ]:
# TODO: build the analysis table with two left joins.
analysis = ...

print(analysis.shape)
analysis.head(3)

In [ ]:
# --- check 4 (do not edit) ---
assert (
    len(analysis) == 2_000
), f"{len(analysis)} rows instead of 2000 — a join multiplied or dropped rows"
assert analysis["bid_id"].is_unique, "bid_id is no longer unique"
assert not [
    c for c in analysis.columns if str(c).endswith(("_x", "_y"))
], "a column name collided: rename the CRM 'name' to 'account_name' before merging"
for col in ["account_name", "country", "qualification_clean"]:
    assert col in analysis.columns, f"{col} is missing from the analysis table"
assert (
    analysis["account_name"].notna().all()
), "every bid belongs to a CRM account — check the join key dtype"
assert (
    analysis["qualification_clean"].isna().sum() == 600
), "600 bids were never qualified: use how='left' so they stay, with a missing qualification"
passed(4)

---

## Done

Four sources read on their own terms, one table at the end, and not a row lost on the way.
That is the whole of data acquisition: **nothing you do later can repair a file you read
wrong.**

| What you did | Where it shows up again |
|---|---|
| Declared the CSV dialect and the dtypes | Lesson 03 — the types you clean are the ones you chose here |
| Flattened a nested payload | Any API you meet after this one |
| Skipped the decoration in a spreadsheet | Every file a human maintains |
| Joined with `how="left"` and checked the row count | Lesson 05, where a lost row becomes a wrong model |